In [ ]:
!pip install -q streamlit networkx pandas matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 29.6 MB/s eta 0:00:00


In [ ]:
%%writefile app.py

import streamlit as st
import json
import sqlite3
import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime


# ============================================================
# PAGE CONFIGURATION
# ============================================================

st.set_page_config(
    page_title="Blackboard AI",
    page_icon="🧠",
    layout="wide",
    initial_sidebar_state="expanded"
)


# ============================================================
# CUSTOM CSS
# ============================================================

st.markdown("""
<style>

.main {
    background-color: #f7f9fc;
}

.block-container {
    padding-top: 2rem;
    padding-bottom: 2rem;
}

[data-testid="stSidebar"] {
    background-color: #111827;
}

[data-testid="stSidebar"] * {
    color: white;
}

.hero {
    padding: 25px;
    border-radius: 18px;
    background: linear-gradient(135deg, #111827, #2563eb);
    color: white;
    margin-bottom: 25px;
}

.hero h1 {
    font-size: 38px;
    margin-bottom: 5px;
}

.hero p {
    font-size: 17px;
    opacity: 0.9;
}

.card {
    background-color: black;
    padding: 20px;
    border-radius: 15px;
    border: 1px solid #e5e7eb;
    margin-bottom: 15px;
}

.agent-card {
    background-color: black;
    padding: 20px;
    border-radius: 15px;
    border: 1px solid #e5e7eb;
    min-height: 190px;
}

.metric-card {
    background-color: black;
    padding: 18px;
    border-radius: 15px;
    border: 1px solid #e5e7eb;
    text-align: center;
}

.metric-number {
    font-size: 30px;
    font-weight: bold;
    color: #2563eb;
}

.metric-label {
    color: #6b7280;
    font-size: 14px;
}

.success-box {
    padding: 20px;
    border-radius: 12px;
    background-color: black;
    border: 1px solid #10b981;
}

.warning-box {
    padding: 20px;
    border-radius: 12px;
    background-color: black;
    border: 1px solid #f59e0b;
}

.danger-box {
    padding: 20px;
    border-radius: 12px;
    background-color: black;
    border: 1px solid #ef4444;
}

.blackboard-box {
    padding: 25px;
    border-radius: 15px;
    background-color: #111827;
    color: white;
    border: 2px solid #2563eb;
    margin-bottom: 20px;
}

.flow-box {
    padding: 18px;
    border-radius: 12px;
    background-color: black;
    border: 1px solid #374151;
    margin-bottom: 10px;

}

</style>
""", unsafe_allow_html=True)


# ============================================================
# LOAD DATA
# ============================================================

with open("blackboard_data.json", "r") as file:
    DATA = json.load(file)


# ============================================================
# AGENT CONFIGURATION
# ============================================================

AGENTS = {

    "Doctor Agent": {
        "icon": "👨‍⚕️",
        "role": "Diagnosis and Treatment",
        "description": "Handles patient symptoms, diagnosis and treatment."
    },

    "Laboratory Agent": {
        "icon": "🧪",
        "role": "Medical Test Results",
        "description": "Handles laboratory tests and medical reports."
    },

    "Pharmacy Agent": {
        "icon": "💊",
        "role": "Medicine Management",
        "description": "Handles medicines and availability."
    },

    "Billing Agent": {
        "icon": "💰",
        "role": "Billing Management",
        "description": "Handles consultation, laboratory and medicine billing."
    }
}

AGENT_NAMES = list(AGENTS.keys())


# ============================================================
# SQLITE DATABASE
# ============================================================

conn = sqlite3.connect(
    "blackboard_communication.db",
    check_same_thread=False
)

cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS communication (

    id INTEGER PRIMARY KEY AUTOINCREMENT,

    agent TEXT,

    action TEXT,

    patient_id TEXT,

    content TEXT,

    timestamp TEXT,

    status TEXT

)
""")

conn.commit()


# ============================================================
# LOG BLACKBOARD ACTIVITY
# ============================================================

def log_activity(
    agent,
    action,
    patient_id,
    content,
    status="Completed"
):

    timestamp = datetime.now().strftime(
        "%Y-%m-%d %H:%M:%S"
    )

    cursor.execute("""
        INSERT INTO communication
        (
            agent,
            action,
            patient_id,
            content,
            timestamp,
            status
        )
        VALUES (?, ?, ?, ?, ?, ?)
    """, (
        agent,
        action,
        patient_id,
        content,
        timestamp,
        status
    ))

    conn.commit()


# ============================================================
# DATA HELPER FUNCTIONS
# ============================================================

def get_patient(patient_id):

    for patient in DATA.get("patients", []):

        if patient.get("patient_id") == patient_id:

            return patient

    return None


def get_agent_record(section, patient_id):

    for record in DATA.get(section, []):

        if record.get("patient_id") == patient_id:

            return record

    return None


def get_doctor(patient_id):

    return get_agent_record(
        "doctor",
        patient_id
    )


def get_laboratory(patient_id):

    return get_agent_record(
        "laboratory",
        patient_id
    )


def get_pharmacy(patient_id):

    return get_agent_record(
        "pharmacy",
        patient_id
    )


def get_billing(patient_id):

    return get_agent_record(
        "billing",
        patient_id
    )


# ============================================================
# CENTRAL BLACKBOARD
# ============================================================

def create_blackboard():

    board = {}

    for patient in DATA["patients"]:

        patient_id = patient["patient_id"]

        board[patient_id] = {

            "patient": patient,

            "doctor": {},

            "laboratory": {},

            "pharmacy": {},

            "billing": {},

            "last_updated":
                datetime.now().strftime(
                    "%Y-%m-%d %H:%M:%S"
                )
        }

    return board


# ============================================================
# INITIALIZE BLACKBOARD
# ============================================================

if "blackboard" not in st.session_state:

    st.session_state.blackboard = create_blackboard()


BLACKBOARD = st.session_state.blackboard


# ============================================================
# WRITE TO BLACKBOARD
# ============================================================

def write_blackboard(
    patient_id,
    section,
    data,
    agent
):

    if patient_id not in BLACKBOARD:

        return False

    BLACKBOARD[patient_id][section] = data

    BLACKBOARD[patient_id]["last_updated"] = (
        datetime.now().strftime(
            "%Y-%m-%d %H:%M:%S"
        )
    )

    log_activity(
        agent,
        "WRITE",
        patient_id,
        json.dumps(data)
    )

    return True


# ============================================================
# READ FROM BLACKBOARD
# ============================================================

def read_blackboard(
    patient_id,
    section,
    agent
):

    if patient_id not in BLACKBOARD:

        return None

    data = BLACKBOARD[patient_id].get(
        section,
        {}
    )

    log_activity(
        agent,
        "READ",
        patient_id,
        json.dumps(data)
    )

    return data


# ============================================================
# DOCTOR AGENT
# ============================================================

def doctor_agent(patient_id):

    patient_data = read_blackboard(
        patient_id,
        "patient",
        "Doctor Agent"
    )

    if not patient_data:

        return None

    symptoms = patient_data.get(
        "symptoms",
        ""
    ).lower()

    if "chest pain" in symptoms:

        diagnosis = "Possible Cardiac Emergency"

        treatment = (
            "Immediate cardiac evaluation"
        )

        priority = "Critical"

        specialist = "Cardiologist"

        follow_up = "Immediate"

    elif "fever" in symptoms:

        diagnosis = "Possible Infection"

        treatment = (
            "Clinical examination and medication"
        )

        priority = "High"

        specialist = "General Physician"

        follow_up = "Within 24 hours"

    elif "headache" in symptoms:

        diagnosis = "Possible Migraine or Viral Condition"

        treatment = (
            "Clinical evaluation and symptomatic treatment"
        )

        priority = "Moderate"

        specialist = "General Physician"

        follow_up = "Within 2 days"

    elif "breathing" in symptoms:

        diagnosis = "Possible Respiratory Condition"

        treatment = (
            "Respiratory evaluation and appropriate therapy"
        )

        priority = "High"

        specialist = "Pulmonologist"

        follow_up = "Immediate evaluation"

    elif "stomach" in symptoms or "vomiting" in symptoms:

        diagnosis = "Possible Gastrointestinal Condition"

        treatment = (
            "Hydration, clinical evaluation and medication"
        )

        priority = "Moderate"

        specialist = "Gastroenterologist"

        follow_up = "Within 24 hours"

    else:

        diagnosis = "General Medical Condition"

        treatment = (
            "Doctor consultation required"
        )

        priority = "Normal"

        specialist = "General Physician"

        follow_up = "As required"

    result = {

        "diagnosis": diagnosis,

        "treatment": treatment,

        "priority": priority,

        "specialist": specialist,

        "follow_up": follow_up

    }

    write_blackboard(
        patient_id,
        "doctor",
        result,
        "Doctor Agent"
    )

    return result


# ============================================================
# LABORATORY AGENT
# ============================================================

def laboratory_agent(patient_id):

    patient_data = read_blackboard(
        patient_id,
        "patient",
        "Laboratory Agent"
    )

    if not patient_data:

        return None

    symptoms = patient_data.get(
        "symptoms",
        ""
    ).lower()

    # First use the actual dataset record.
    existing = get_laboratory(patient_id)

    if existing:

        result = dict(existing)

        result.pop(
            "patient_id",
            None
        )

    elif "chest pain" in symptoms:

        result = {

            "test": "ECG + Troponin",

            "ecg": "Abnormal",

            "troponin": "Elevated",

            "blood_pressure": "150/95 mmHg",

            "test_status": "Completed"

        }

    elif "fever" in symptoms:

        result = {

            "test": "Complete Blood Count",

            "white_blood_cells": "Elevated",

            "infection_marker": "Positive",

            "temperature": "102°F",

            "test_status": "Completed"

        }

    elif "headache" in symptoms:

        result = {

            "test": "CBC + Blood Pressure",

            "white_blood_cells": "Normal",

            "blood_pressure": "135/85 mmHg",

            "hemoglobin": "13.8 g/dL",

            "test_status": "Completed"

        }

    else:

        result = {

            "test": "Routine Blood Test",

            "hemoglobin": "Normal",

            "white_blood_cells": "Normal",

            "blood_sugar": "Normal",

            "test_status": "Completed"

        }

    write_blackboard(
        patient_id,
        "laboratory",
        result,
        "Laboratory Agent"
    )

    return result


# ============================================================
# PHARMACY AGENT
# ============================================================

def pharmacy_agent(patient_id):

    doctor_data = read_blackboard(
        patient_id,
        "doctor",
        "Pharmacy Agent"
    )

    if not doctor_data:

        doctor_data = doctor_agent(
            patient_id
        )

    if not doctor_data:

        return None

    # Use actual dataset pharmacy information
    existing = get_pharmacy(patient_id)

    if existing:

        result = dict(existing)

        result.pop(
            "patient_id",
            None
        )

    else:

        diagnosis = doctor_data.get(
            "diagnosis",
            ""
        )

        if "Cardiac" in diagnosis:

            result = {

                "medicine": "Cardiac Emergency Medication",

                "availability": "Available",

                "quantity": 10,

                "dosage": "As prescribed by cardiologist",

                "pharmacy_status": "Ready"

            }

        elif "Infection" in diagnosis:

            result = {

                "medicine": "Antibiotic",

                "availability": "Available",

                "quantity": 20,

                "dosage": "As prescribed by doctor",

                "pharmacy_status": "Ready"

            }

        elif "Migraine" in diagnosis:

            result = {

                "medicine": "Migraine Symptomatic Medicine",

                "availability": "Available",

                "quantity": 15,

                "dosage": "As prescribed by doctor",

                "pharmacy_status": "Ready"

            }

        else:

            result = {

                "medicine": "General Medicine",

                "availability": "Available",

                "quantity": 10,

                "dosage": "As prescribed by doctor",

                "pharmacy_status": "Ready"

            }

    write_blackboard(
        patient_id,
        "pharmacy",
        result,
        "Pharmacy Agent"
    )

    return result


# ============================================================
# BILLING AGENT
# ============================================================

def billing_agent(patient_id):

    patient = get_patient(patient_id)

    if not patient:

        return None

    # IMPORTANT:
    # Billing data is stored in DATA["billing"],
    # NOT inside DATA["patients"].

    billing_data = get_billing(
        patient_id
    )

    laboratory_data = read_blackboard(
        patient_id,
        "laboratory",
        "Billing Agent"
    )

    pharmacy_data = read_blackboard(
        patient_id,
        "pharmacy",
        "Billing Agent"
    )

    if billing_data:

        consultation = billing_data.get(
            "consultation_fee",
            0
        )

        lab_cost = billing_data.get(
            "laboratory_fee",
            0
        )

        medicine_cost = billing_data.get(
            "medicine_fee",
            0
        )

        room_charge = billing_data.get(
            "room_charge",
            0
        )

        total = billing_data.get(
            "total_amount",
            consultation +
            lab_cost +
            medicine_cost +
            room_charge
        )

        payment_status = billing_data.get(
            "payment_status",
            "Unknown"
        )

    else:

        consultation = 0

        lab_cost = 0

        medicine_cost = 0

        room_charge = 0

        total = 0

        payment_status = "Unknown"

    result = {

        "consultation": consultation,

        "laboratory": lab_cost,

        "medicine": medicine_cost,

        "room_charge": room_charge,

        "total": total,

        "payment_status": payment_status

    }

    write_blackboard(
        patient_id,
        "billing",
        result,
        "Billing Agent"
    )

    return result


# ============================================================
# COMPLETE BLACKBOARD PROCESS
# ============================================================

def run_blackboard_process(patient_id):

    results = {}

    results["doctor"] = doctor_agent(
        patient_id
    )

    results["laboratory"] = laboratory_agent(
        patient_id
    )

    results["pharmacy"] = pharmacy_agent(
        patient_id
    )

    results["billing"] = billing_agent(
        patient_id
    )

    return results


# ============================================================
# RESET BLACKBOARD FOR ONE PATIENT
# ============================================================

def reset_patient_blackboard(patient_id):

    patient = get_patient(
        patient_id
    )

    if not patient:

        return False

    BLACKBOARD[patient_id] = {

        "patient": patient,

        "doctor": {},

        "laboratory": {},

        "pharmacy": {},

        "billing": {},

        "last_updated":
            datetime.now().strftime(
                "%Y-%m-%d %H:%M:%S"
            )
    }

    return True


# ============================================================
# NORMALIZE QUESTION
# ============================================================

def normalize_question(query):

    q = query.lower().strip()

    replacements = {

        "?": " ",
        ",": " ",
        ".": " ",
        "!": " ",
        ":": " ",
        ";": " ",
        "'": " ",
        '"': " "

    }

    for old, new in replacements.items():

        q = q.replace(
            old,
            new
        )

    return " ".join(
        q.split()
    )


# ============================================================
# QUESTION TYPE DETECTION
# ============================================================

def detect_question_type(query):

    q = normalize_question(
        query
    )

    # COMPLETE
    complete_words = [

        "complete",
        "everything",
        "all information",
        "all details",
        "full information",
        "full details",
        "complete information",
        "complete details",
        "entire information",
        "entire record",
        "full report",
        "complete report",
        "whole record",
        "all data",
        "entire case"

    ]

    if any(
        word in q
        for word in complete_words
    ):

        return "complete"


    # PATIENT
    patient_words = [

        "patient",
        "name",
        "age",
        "gender",
        "sex",
        "blood group",
        "blood type",
        "symptom",
        "symptoms",
        "city",
        "location",
        "profile",
        "personal details",
        "basic details"

    ]

    # BILLING
    billing_words = [

        "bill",
        "billing",
        "amount",
        "cost",
        "price",
        "fee",
        "fees",
        "payment",
        "paid",
        "pay",
        "charge",
        "charges",
        "money",
        "expense",
        "expenses",
        "outstanding"

    ]

    # PHARMACY
    pharmacy_words = [

        "medicine",
        "medicines",
        "medication",
        "medications",
        "drug",
        "drugs",
        "pharmacy",
        "tablet",
        "tablets",
        "dosage",
        "dose",
        "prescription",
        "prescribed",
        "stock",
        "available",
        "availability",
        "dispensed",
        "reserved"

    ]

    # LAB
    laboratory_words = [

        "lab",
        "laboratory",
        "test",
        "tests",
        "result",
        "results",
        "report",
        "reports",
        "ecg",
        "troponin",
        "blood pressure",
        "temperature",
        "oxygen",
        "oxygen level",
        "white blood",
        "hemoglobin",
        "amylase",
        "peak flow",
        "investigation"

    ]

    # DOCTOR
    doctor_words = [

        "diagnosis",
        "diagnose",
        "condition",
        "disease",
        "doctor",
        "treatment",
        "treat",
        "specialist",
        "severity",
        "severe",
        "critical",
        "priority",
        "urgent",
        "urgency",
        "follow up",
        "follow-up",
        "medical condition"

    ]

    # Cross agent detection

    has_billing = any(
        word in q
        for word in billing_words
    )

    has_pharmacy = any(
        word in q
        for word in pharmacy_words
    )

    has_lab = any(
        word in q
        for word in laboratory_words
    )

    has_doctor = any(
        word in q
        for word in doctor_words
    )

    # Cross-agent questions

    if (
        has_doctor
        and has_pharmacy
        and has_billing
    ):

        return "cross"

    if (
        has_lab
        and has_pharmacy
    ):

        return "cross"

    if (
        has_doctor
        and has_lab
    ):

        return "cross"

    if (
        has_doctor
        and has_pharmacy
    ):

        return "cross"

    if (
        has_lab
        and has_billing
    ):

        return "cross"

    if (
        has_pharmacy
        and has_billing
    ):

        return "cross"

    if has_billing:

        return "billing"

    if has_pharmacy:

        return "pharmacy"

    if has_lab:

        return "laboratory"

    if has_doctor:

        return "doctor"

    if any(
        word in q
        for word in patient_words
    ):

        return "patient"

    return "unknown"


# ============================================================
# SPECIFIC PATIENT ANSWER
# ============================================================

def answer_patient_question(
    patient_id,
    query
):

    patient = get_patient(
        patient_id
    )

    if not patient:

        return "Patient not found."

    q = normalize_question(
        query
    )

    log_activity(
        "Doctor Agent",
        "READ",
        patient_id,
        "Patient information requested."
    )

    # NAME
    if (
        "name" in q
        or "who is the patient" in q
        or "patient name" in q
    ):

        return (
            f"The patient's name is "
            f"**{patient['name']}**."
        )

    # AGE
    if (
        "age" in q
        or "how old" in q
        or "old is" in q
    ):

        return (
            f"The patient is "
            f"**{patient['age']} years old**."
        )

    # GENDER
    if (
        "gender" in q
        or "sex" in q
    ):

        return (
            f"The patient's gender is "
            f"**{patient['gender']}**."
        )

    # BLOOD GROUP
    if (
        "blood group" in q
        or "blood type" in q
        or "blood group is" in q
    ):

        return (
            f"The patient's blood group is "
            f"**{patient['blood_group']}**."
        )

    # SYMPTOMS
    if (
        "symptom" in q
        or "symptoms" in q
        or "problem" in q
        or "problems" in q
    ):

        return (
            f"The patient's symptoms are "
            f"**{patient['symptoms']}**."
        )

    # CITY
    if (
        "city" in q
        or "where does" in q
        or "location" in q
        or "live" in q
    ):

        return (
            f"The patient is from "
            f"**{patient['city']}**."
        )

    # ID
    if (
        "id" in q
        or "patient id" in q
    ):

        return (
            f"The patient ID is "
            f"**{patient['patient_id']}**."
        )

    return (
        f"**Patient Information**\n\n"
        f"Name: {patient['name']}\n\n"
        f"Age: {patient['age']} years\n\n"
        f"Gender: {patient['gender']}\n\n"
        f"Blood Group: {patient['blood_group']}\n\n"
        f"Symptoms: {patient['symptoms']}\n\n"
        f"City: {patient['city']}"
    )


# ============================================================
# SPECIFIC DOCTOR ANSWER
# ============================================================

def answer_doctor_question(
    patient_id,
    query
):

    doctor = get_doctor(
        patient_id
    )

    if not doctor:

        doctor = doctor_agent(
            patient_id
        )

    if not doctor:

        return "Doctor information is not available."

    q = normalize_question(
        query
    )

    log_activity(
        "Doctor Agent",
        "READ",
        patient_id,
        "Doctor information requested."
    )

    # DIAGNOSIS
    if (
        "diagnosis" in q
        or "diagnose" in q
        or "condition" in q
        or "disease" in q
        or "what is wrong" in q
        or "what does the patient have" in q
    ):

        return (
            f"The diagnosis for this patient is "
            f"**{doctor.get('diagnosis', 'Not available')}**."
        )

    # TREATMENT
    if (
        "treatment" in q
        or "treat" in q
        or "what should be done" in q
        or "how should" in q
    ):

        return (
            f"The recommended treatment is "
            f"**{doctor.get('treatment', 'Not available')}**."
        )

    # DOCTOR NAME
    if (
        "doctor name" in q
        or "who is the doctor" in q
        or "who is treating" in q
    ):

        original = get_doctor(
            patient_id
        )

        if original:

            return (
                f"The doctor handling this patient is "
                f"**{original.get('doctor_name', 'Not available')}**."
            )

        return "Doctor name is not available."

    # DEPARTMENT
    if (
        "department" in q
        or "which department" in q
    ):

        original = get_doctor(
            patient_id
        )

        if original:

            return (
                f"The patient is being handled by the "
                f"**{original.get('department', 'Not available')}** department."
            )

        return "Department information is not available."

    # SPECIALIST
    if (
        "specialist" in q
        or "which specialist" in q
    ):

        return (
            f"The recommended specialist is "
            f"**{doctor.get('specialist', 'Not available')}**."
        )

    # SEVERITY
    if (
        "severity" in q
        or "serious" in q
        or "critical" in q
        or "priority" in q
        or "how serious" in q
    ):

        return (
            f"The patient's priority/severity is "
            f"**{doctor.get('priority', 'Not available')}**."
        )

    # FOLLOW UP
    if (
        "follow up" in q
        or "follow-up" in q
        or "when should" in q
        or "review" in q
    ):

        return (
            f"The recommended follow-up is "
            f"**{doctor.get('follow_up', 'Not available')}**."
        )

    # URGENT
    if (
        "urgent" in q
        or "urgency" in q
        or "emergency" in q
        or "immediate" in q
    ):

        priority = doctor.get(
            "priority",
            ""
        )

        follow_up = doctor.get(
            "follow_up",
            ""
        )

        return (
            f"The case priority is **{priority}** "
            f"and the recommended follow-up is "
            f"**{follow_up}**."
        )

    return (
        f"**Doctor Information**\n\n"
        f"Diagnosis: {doctor.get('diagnosis', 'Not available')}\n\n"
        f"Treatment: {doctor.get('treatment', 'Not available')}\n\n"
        f"Priority: {doctor.get('priority', 'Not available')}\n\n"
        f"Specialist: {doctor.get('specialist', 'Not available')}\n\n"
        f"Follow-up: {doctor.get('follow_up', 'Not available')}"
    )


# ============================================================
# SPECIFIC LABORATORY ANSWER
# ============================================================

def answer_laboratory_question(
    patient_id,
    query
):

    laboratory = get_laboratory(
        patient_id
    )

    if not laboratory:

        laboratory = laboratory_agent(
            patient_id
        )

    if not laboratory:

        return "Laboratory information is not available."

    q = normalize_question(
        query
    )

    log_activity(
        "Laboratory Agent",
        "READ",
        patient_id,
        "Laboratory information requested."
    )

    # TEST
    if (
        "test name" in q
        or "which test" in q
        or "what test" in q
        or "test performed" in q
        or "test was done" in q
        or "investigation" in q
    ):

        test_name = laboratory.get(
            "test_name",
            laboratory.get(
                "test",
                "Not available"
            )
        )

        return (
            f"The laboratory test performed was "
            f"**{test_name}**."
        )

    # RESULT
    if (
        "result" in q
        or "results" in q
        or "report" in q
        or "lab report" in q
        or "what did the test show" in q
    ):

        result = laboratory.get(
            "result",
            "Not available"
        )

        return (
            f"The laboratory result is "
            f"**{result}**."
        )

    # DATE
    if (
        "date" in q
        or "when was the test" in q
        or "test date" in q
    ):

        return (
            f"The laboratory test was performed on "
            f"**{laboratory.get('test_date', 'Not available')}**."
        )

    # BLOOD PRESSURE
    if (
        "blood pressure" in q
        or "bp" in q
    ):

        value = laboratory.get(
            "blood_pressure",
            "Not available"
        )

        return (
            f"The patient's blood pressure is "
            f"**{value}**."
        )

    # TEMPERATURE
    if "temperature" in q:

        return (
            f"The patient's temperature is "
            f"**{laboratory.get('temperature', 'Not available')}**."
        )

    # OXYGEN
    if (
        "oxygen" in q
        or "spo2" in q
    ):

        return (
            f"The patient's oxygen level is "
            f"**{laboratory.get('oxygen_level', 'Not available')}**."
        )

    # TROPONIN
    if "troponin" in q:

        return (
            f"The patient's troponin result is "
            f"**{laboratory.get('troponin', 'Not available')}**."
        )

    # ECG
    if "ecg" in q:

        return (
            f"The ECG result is "
            f"**{laboratory.get('ecg', 'Not available')}**."
        )

    # WBC
    if (
        "white blood" in q
        or "white blood cells" in q
        or "wbc" in q
    ):

        return (
            f"The white blood cell count is "
            f"**{laboratory.get('white_blood_cells', 'Not available')}**."
        )

    # HEMOGLOBIN
    if (
        "hemoglobin" in q
        or "haemoglobin" in q
    ):

        return (
            f"The hemoglobin level is "
            f"**{laboratory.get('hemoglobin', 'Not available')}**."
        )

    # PEAK FLOW
    if "peak flow" in q:

        return (
            f"The peak flow is "
            f"**{laboratory.get('peak_flow', 'Not available')}**."
        )

    # AMYLASE
    if "amylase" in q:

        return (
            f"The amylase level is "
            f"**{laboratory.get('amylase', 'Not available')}**."
        )

    # STATUS
    if (
        "status" in q
        or "completed" in q
        or "reviewed" in q
        or "monitoring" in q
    ):

        return (
            f"The laboratory test status is "
            f"**{laboratory.get('test_status', 'Not available')}**."
        )

    # ABNORMAL
    if (
        "normal" in q
        or "abnormal" in q
        or "anything unusual" in q
        or "anything wrong" in q
    ):

        return (
            f"The laboratory report shows "
            f"**{laboratory.get('result', 'Not available')}**."
        )

    return (
        f"**Laboratory Information**\n\n"
        f"Test: {laboratory.get('test_name', laboratory.get('test', 'Not available'))}\n\n"
        f"Result: {laboratory.get('result', 'Not available')}\n\n"
        f"Status: {laboratory.get('test_status', 'Not available')}"
    )


# ============================================================
# SPECIFIC PHARMACY ANSWER
# ============================================================

def answer_pharmacy_question(
    patient_id,
    query
):

    pharmacy = get_pharmacy(
        patient_id
    )

    if not pharmacy:

        pharmacy = pharmacy_agent(
            patient_id
        )

    if not pharmacy:

        return "Pharmacy information is not available."

    q = normalize_question(
        query
    )

    log_activity(
        "Pharmacy Agent",
        "READ",
        patient_id,
        "Pharmacy information requested."
    )

    # MEDICINE
    if (
        "medicine" in q
        or "medication" in q
        or "drug" in q
        or "what is prescribed" in q
        or "what was prescribed" in q
    ):

        return (
            f"The medicine for this patient is "
            f"**{pharmacy.get('medicine', 'Not available')}**."
        )

    # DOSAGE
    if (
        "dosage" in q
        or "dose" in q
        or "strength" in q
    ):

        return (
            f"The prescribed dosage is "
            f"**{pharmacy.get('dosage', 'Not available')}**."
        )

    # QUANTITY
    if (
        "quantity" in q
        or "how many" in q
        or "number of medicines" in q
        or "how much medicine" in q
    ):

        return (
            f"The available quantity is "
            f"**{pharmacy.get('quantity', 'Not available')}**."
        )

    # AVAILABILITY
    if (
        "available" in q
        or "availability" in q
        or "stock" in q
        or "in stock" in q
    ):

        return (
            f"The medicine availability is "
            f"**{pharmacy.get('availability', 'Not available')}**."
        )

    # PRESCRIPTION
    if (
        "prescription" in q
        or "verified" in q
        or "approved" in q
    ):

        return (
            f"The prescription status is "
            f"**{pharmacy.get('prescription_status', 'Not available')}**."
        )

    # PHARMACY STATUS
    if (
        "pharmacy status" in q
        or "dispensed" in q
        or "ready" in q
        or "reserved" in q
    ):

        return (
            f"The pharmacy status is "
            f"**{pharmacy.get('pharmacy_status', 'Not available')}**."
        )

    return (
        f"**Pharmacy Information**\n\n"
        f"Medicine: {pharmacy.get('medicine', 'Not available')}\n\n"
        f"Dosage: {pharmacy.get('dosage', 'Not available')}\n\n"
        f"Quantity: {pharmacy.get('quantity', 'Not available')}\n\n"
        f"Availability: {pharmacy.get('availability', 'Not available')}\n\n"
        f"Prescription Status: {pharmacy.get('prescription_status', 'Not available')}\n\n"
        f"Pharmacy Status: {pharmacy.get('pharmacy_status', 'Not available')}"
    )


# ============================================================
# SPECIFIC BILLING ANSWER
# ============================================================

def answer_billing_question(
    patient_id,
    query
):

    billing = get_billing(
        patient_id
    )

    if not billing:

        billing = billing_agent(
            patient_id
        )

        if not billing:

            return "Billing information is not available."

    q = normalize_question(
        query
    )

    log_activity(
        "Billing Agent",
        "READ",
        patient_id,
        "Billing information requested."
    )

    consultation = billing.get(
        "consultation_fee",
        billing.get(
            "consultation",
            0
        )
    )

    laboratory = billing.get(
        "laboratory_fee",
        billing.get(
            "laboratory",
            0
        )
    )

    medicine = billing.get(
        "medicine_fee",
        billing.get(
            "medicine",
            0
        )
    )

    room = billing.get(
        "room_charge",
        0
    )

    total = billing.get(
        "total_amount",
        billing.get(
            "total",
            consultation +
            laboratory +
            medicine +
            room
        )
    )

    payment = billing.get(
        "payment_status",
        "Unknown"
    )

    # TOTAL
    if (
        "total" in q
        or "total bill" in q
        or "total amount" in q
        or "final cost" in q
        or "how much" in q
        or "how much does" in q
        or "how much money" in q
        or "overall cost" in q
        or "final amount" in q
    ):

        return (
            f"The total bill for **{patient_id}** is "
            f"**₹{total:,}**."
        )

    # CONSULTATION
    if (
        "consultation" in q
        or "doctor fee" in q
        or "doctor fees" in q
    ):

        return (
            f"The consultation fee is "
            f"**₹{consultation:,}**."
        )

    # LABORATORY COST
    if (
        "laboratory fee" in q
        or "lab fee" in q
        or "lab cost" in q
        or "laboratory cost" in q
        or "test cost" in q
    ):

        return (
            f"The laboratory fee is "
            f"**₹{laboratory:,}**."
        )

    # MEDICINE COST
    if (
        "medicine fee" in q
        or "medicine cost" in q
        or "pharmacy cost" in q
        or "drug cost" in q
    ):

        return (
            f"The medicine fee is "
            f"**₹{medicine:,}**."
        )

    # ROOM
    if (
        "room" in q
        or "room charge" in q
        or "room cost" in q
    ):

        return (
            f"The room charge is "
            f"**₹{room:,}**."
        )

    # PAYMENT
    if (
        "payment" in q
        or "paid" in q
        or "pay" in q
        or "outstanding" in q
    ):

        if payment.lower() == "paid":

            return (
                f"The bill status is **Paid**. "
                f"The total amount was **₹{total:,}**."
            )

        elif payment.lower() == "pending":

            return (
                f"The payment is **Pending**. "
                f"The total bill is **₹{total:,}**."
            )

        return (
            f"The payment status is "
            f"**{payment}**."
        )

    # BREAKDOWN
    if (
        "breakdown" in q
        or "charges" in q
        or "billing details" in q
        or "billing information" in q
        or "explain the bill" in q
        or "show the bill" in q
    ):

        return (
            f"**Billing Breakdown for {patient_id}**\n\n"
            f"Consultation Fee: **₹{consultation:,}**\n\n"
            f"Laboratory Fee: **₹{laboratory:,}**\n\n"
            f"Medicine Fee: **₹{medicine:,}**\n\n"
            f"Room Charge: **₹{room:,}**\n\n"
            f"Total Amount: **₹{total:,}**\n\n"
            f"Payment Status: **{payment}**"
        )

    return (
        f"The total bill for **{patient_id}** is "
        f"**₹{total:,}**."
    )


# ============================================================
# CROSS-AGENT ANSWER
# ============================================================

def answer_cross_question(
    patient_id,
    query
):

    q = normalize_question(
        query
    )

    patient = get_patient(
        patient_id
    )

    doctor = get_doctor(
        patient_id
    )

    laboratory = get_laboratory(
        patient_id
    )

    pharmacy = get_pharmacy(
        patient_id
    )

    billing = get_billing(
        patient_id
    )

    if not patient:

        return "Patient information is not available."

    log_activity(
        "Blackboard Coordinator",
        "READ",
        patient_id,
        "Cross-agent information requested."
    )

    # DOCTOR + LAB
    if (
        ("diagnosis" in q or "condition" in q)
        and
        ("lab" in q or "test" in q or "result" in q)
    ):

        diagnosis = doctor.get(
            "diagnosis",
            "Not available"
        ) if doctor else "Not available"

        lab_result = laboratory.get(
            "result",
            "Not available"
        ) if laboratory else "Not available"

        return (
            f"**Doctor + Laboratory Information**\n\n"
            f"Diagnosis: **{diagnosis}**\n\n"
            f"Laboratory Result: **{lab_result}**"
        )

    # DOCTOR + PHARMACY
    if (
        ("diagnosis" in q or "treatment" in q)
        and
        ("medicine" in q or "pharmacy" in q)
    ):

        diagnosis = doctor.get(
            "diagnosis",
            "Not available"
        ) if doctor else "Not available"

        treatment = doctor.get(
            "treatment",
            "Not available"
        ) if doctor else "Not available"

        medicine = pharmacy.get(
            "medicine",
            "Not available"
        ) if pharmacy else "Not available"

        return (
            f"**Doctor + Pharmacy Information**\n\n"
            f"Diagnosis: **{diagnosis}**\n\n"
            f"Treatment: **{treatment}**\n\n"
            f"Medicine: **{medicine}**"
        )

    # LAB + PHARMACY
    if (
        ("lab" in q or "test" in q or "result" in q)
        and
        ("medicine" in q or "pharmacy" in q)
    ):

        lab_result = laboratory.get(
            "result",
            "Not available"
        ) if laboratory else "Not available"

        medicine = pharmacy.get(
            "medicine",
            "Not available"
        ) if pharmacy else "Not available"

        return (
            f"**Laboratory + Pharmacy Information**\n\n"
            f"Laboratory Result: **{lab_result}**\n\n"
            f"Medicine: **{medicine}**"
        )

    # LAB + BILLING
    if (
        ("lab" in q or "test" in q)
        and
        ("bill" in q or "cost" in q or "fee" in q)
    ):

        test_name = laboratory.get(
            "test_name",
            "Not available"
        ) if laboratory else "Not available"

        lab_result = laboratory.get(
            "result",
            "Not available"
        ) if laboratory else "Not available"

        lab_fee = billing.get(
            "laboratory_fee",
            0
        ) if billing else 0

        return (
            f"**Laboratory + Billing Information**\n\n"
            f"Test: **{test_name}**\n\n"
            f"Result: **{lab_result}**\n\n"
            f"Laboratory Fee: **₹{lab_fee:,}**"
        )

    # PHARMACY + BILLING
    if (
        ("medicine" in q or "pharmacy" in q)
        and
        ("bill" in q or "cost" in q or "fee" in q)
    ):

        medicine_name = pharmacy.get(
            "medicine",
            "Not available"
        ) if pharmacy else "Not available"

        medicine_fee = billing.get(
            "medicine_fee",
            0
        ) if billing else 0

        return (
            f"**Pharmacy + Billing Information**\n\n"
            f"Medicine: **{medicine_name}**\n\n"
            f"Medicine Fee: **₹{medicine_fee:,}**"
        )

    # DOCTOR + BILLING
    if (
        ("diagnosis" in q or "treatment" in q)
        and
        ("bill" in q or "cost" in q or "amount" in q)
    ):

        diagnosis = doctor.get(
            "diagnosis",
            "Not available"
        ) if doctor else "Not available"

        treatment = doctor.get(
            "treatment",
            "Not available"
        ) if doctor else "Not available"

        total = billing.get(
            "total_amount",
            0
        ) if billing else 0

        return (
            f"**Doctor + Billing Information**\n\n"
            f"Diagnosis: **{diagnosis}**\n\n"
            f"Treatment: **{treatment}**\n\n"
            f"Total Bill: **₹{total:,}**"
        )

    # GENERAL CROSS AGENT
    diagnosis = doctor.get(
        "diagnosis",
        "Not available"
    ) if doctor else "Not available"

    test_name = laboratory.get(
        "test_name",
        laboratory.get(
            "test",
            "Not available"
        )
    ) if laboratory else "Not available"

    lab_result = laboratory.get(
        "result",
        "Not available"
    ) if laboratory else "Not available"

    medicine = pharmacy.get(
        "medicine",
        "Not available"
    ) if pharmacy else "Not available"

    total = billing.get(
        "total_amount",
        0
    ) if billing else 0

    return (
        f"**Patient Case Summary**\n\n"
        f"Patient: **{patient.get('name', 'Not available')}**\n\n"
        f"Diagnosis: **{diagnosis}**\n\n"
        f"Laboratory Test: **{test_name}**\n\n"
        f"Laboratory Result: **{lab_result}**\n\n"
        f"Medicine: **{medicine}**\n\n"
        f"Total Bill: **₹{total:,}**"
    )


# ============================================================
# COMPLETE HUMAN-READABLE SUMMARY
# ============================================================

def complete_summary(patient_id):

    patient = get_patient(
        patient_id
    )

    doctor = get_doctor(
        patient_id
    )

    laboratory = get_laboratory(
        patient_id
    )

    pharmacy = get_pharmacy(
        patient_id
    )

    billing = get_billing(
        patient_id
    )

    if not patient:

        return "Patient not found."

    diagnosis = (
        doctor.get(
            "diagnosis",
            "Not available"
        )
        if doctor
        else "Not available"
    )

    treatment = (
        doctor.get(
            "treatment",
            "Not available"
        )
        if doctor
        else "Not available"
    )

    test_name = (
        laboratory.get(
            "test_name",
            laboratory.get(
                "test",
                "Not available"
            )
        )
        if laboratory
        else "Not available"
    )

    lab_result = (
        laboratory.get(
            "result",
            "Not available"
        )
        if laboratory
        else "Not available"
    )

    medicine = (
        pharmacy.get(
            "medicine",
            "Not available"
        )
        if pharmacy
        else "Not available"
    )

    total = (
        billing.get(
            "total_amount",
            0
        )
        if billing
        else 0
    )

    payment = (
        billing.get(
            "payment_status",
            "Unknown"
        )
        if billing
        else "Unknown"
    )

    return (
        f"### Complete Patient Summary\n\n"
        f"**Patient:** {patient['name']} ({patient['patient_id']})\n\n"
        f"**Age:** {patient['age']} years\n\n"
        f"**Gender:** {patient['gender']}\n\n"
        f"**Blood Group:** {patient['blood_group']}\n\n"
        f"**Symptoms:** {patient['symptoms']}\n\n"
        f"**City:** {patient['city']}\n\n"
        f"---\n\n"
        f"### 👨‍⚕️ Doctor\n\n"
        f"**Diagnosis:** {diagnosis}\n\n"
        f"**Treatment:** {treatment}\n\n"
        f"---\n\n"
        f"### 🧪 Laboratory\n\n"
        f"**Test:** {test_name}\n\n"
        f"**Result:** {lab_result}\n\n"
        f"---\n\n"
        f"### 💊 Pharmacy\n\n"
        f"**Medicine:** {medicine}\n\n"
        f"---\n\n"
        f"### 💰 Billing\n\n"
        f"**Total Amount:** ₹{total:,}\n\n"
        f"**Payment Status:** {payment}"
    )


# ============================================================
# QUERY PROCESSOR
# ============================================================

def process_query(
    patient_id,
    query
):

    if patient_id not in BLACKBOARD:

        return {

            "type": "error",

            "message":
                "Patient ID not found."

        }

    question_type = detect_question_type(
        query
    )

    # COMPLETE
    if question_type == "complete":

        run_blackboard_process(
            patient_id
        )

        return {

            "type": "complete",

            "answer":
                complete_summary(patient_id),

            "data":
                BLACKBOARD[patient_id]

        }

    # PATIENT
    if question_type == "patient":

        return {

            "type": "patient",

            "answer":
                answer_patient_question(
                    patient_id,
                    query
                )

        }

    # DOCTOR
    if question_type == "doctor":

        # Ensure Doctor Agent has written information.
        if not BLACKBOARD[patient_id]["doctor"]:

            doctor_agent(
                patient_id
            )

        return {

            "type": "doctor",

            "answer":
                answer_doctor_question(
                    patient_id,
                    query
                )

        }

    # LAB
    if question_type == "laboratory":

        if not BLACKBOARD[patient_id]["laboratory"]:

            laboratory_agent(
                patient_id
            )

        return {

            "type": "laboratory",

            "answer":
                answer_laboratory_question(
                    patient_id,
                    query
                )

        }

    # PHARMACY
    if question_type == "pharmacy":

        if not BLACKBOARD[patient_id]["pharmacy"]:

            pharmacy_agent(
                patient_id
            )

        return {

            "type": "pharmacy",

            "answer":
                answer_pharmacy_question(
                    patient_id,
                    query
                )

        }

    # BILLING
    if question_type == "billing":

        if not BLACKBOARD[patient_id]["billing"]:

            billing_agent(
                patient_id
            )

        return {

            "type": "billing",

            "answer":
                answer_billing_question(
                    patient_id,
                    query
                )

        }

    # CROSS AGENT
    if question_type == "cross":

        run_blackboard_process(
            patient_id
        )

        return {

            "type": "cross",

            "answer":
                answer_cross_question(
                    patient_id,
                    query
                )

        }

    return {

        "type": "unknown",

        "message":
            "I could not identify the required information. "
            "Try asking about the patient, diagnosis, treatment, "
            "laboratory, medicine, pharmacy, billing or complete information."

    }


# ============================================================
# SIDEBAR
# ============================================================

st.sidebar.markdown(
    """
    <div style="
        text-align:center;
        padding:15px;
        font-size:28px;
        font-weight:bold;
    ">
        🧠 Blackboard
    </div>
    """,
    unsafe_allow_html=True
)


st.sidebar.markdown(
    "### Navigation"
)


menu = [

    "🏠 Dashboard",

    "🧠 Ask Blackboard",

    "📋 Blackboard",

    "👨‍⚕️ Doctor Agent",

    "🧪 Laboratory Agent",

    "💊 Pharmacy Agent",

    "💰 Billing Agent",

    "📡 Agent Activity",

    "🌐 Architecture",

    "📜 Communication History"

]


choice = st.sidebar.radio(
    "Go to",
    menu
)


# ============================================================
# DASHBOARD
# ============================================================

if choice == "🏠 Dashboard":

    st.markdown("""
    <div class="hero">

        <h1>🧠 Blackboard Architecture</h1>

        <p>
        Centralized Knowledge Sharing Multi-Agent System
        </p>

        <p>
        Multiple independent agents read and write
        information through a common Blackboard.
        </p>

    </div>
    """, unsafe_allow_html=True)


    col1, col2, col3, col4 = st.columns(4)


    with col1:

        st.markdown("""
        <div class="metric-card">

        <div class="metric-number">
        4
        </div>

        <div class="metric-label">
        Active Agents
        </div>

        </div>
        """, unsafe_allow_html=True)


    with col2:

        st.markdown(f"""
        <div class="metric-card">

        <div class="metric-number">
        {len(DATA["patients"])}
        </div>

        <div class="metric-label">
        Patients
        </div>

        </div>
        """, unsafe_allow_html=True)


    with col3:

        st.markdown("""
        <div class="metric-card">

        <div class="metric-number">
        1
        </div>

        <div class="metric-label">
        Central Blackboard
        </div>

        </div>
        """, unsafe_allow_html=True)


    with col4:

        st.markdown("""
        <div class="metric-card">

        <div class="metric-number">
        LIVE
        </div>

        <div class="metric-label">
        Knowledge Store
        </div>

        </div>
        """, unsafe_allow_html=True)


    st.markdown("## 🤖 Agents")


    cols = st.columns(4)


    for index, (name, agent) in enumerate(
        AGENTS.items()
    ):

        with cols[index]:

            st.markdown(
                f"""
                <div class="agent-card">

                <h2>
                {agent["icon"]}
                </h2>

                <h3>
                {name}
                </h3>

                <p>
                <b>{agent["role"]}</b>
                </p>

                <p>
                {agent["description"]}
                </p>

                </div>
                """,
                unsafe_allow_html=True
            )


    st.markdown("## 🔄 Blackboard Process")


    st.markdown("""
    <div class="blackboard-box">

    <h2>🧠 CENTRAL BLACKBOARD</h2>

    <p>
    Shared knowledge repository
    </p>

    </div>
    """, unsafe_allow_html=True)


    st.markdown("""
    <div class="flow-box">

    👨‍⚕️ Doctor Agent → READ / WRITE → Blackboard

    </div>

    <div class="flow-box">

    🧪 Laboratory Agent → READ / WRITE → Blackboard

    </div>

    <div class="flow-box">

    💊 Pharmacy Agent → READ / WRITE → Blackboard

    </div>

    <div class="flow-box">

    💰 Billing Agent → READ / WRITE → Blackboard

    </div>
    """, unsafe_allow_html=True)


# ============================================================
# ASK BLACKBOARD
# ============================================================

elif choice == "🧠 Ask Blackboard":

    st.header("🧠 Ask Blackboard")


    st.write(
        "Ask a question about a patient. "
        "Agents retrieve information through "
        "the central Blackboard."
    )


    patient_id = st.text_input(
        "Patient ID",
        placeholder="Example: P101"
    ).strip().upper()


    query = st.text_area(
        "Your Question",
        placeholder=(
            "Example: What is the patient's diagnosis?\n"
            "Example: What is Ravi Kumar's age?\n"
            "Example: What are the laboratory results?\n"
            "Example: What medicine is available?\n"
            "Example: What is the total bill?\n"
            "Example: Give me complete information."
        )
    )


    if st.button(
        "🚀 Ask Blackboard",
        use_container_width=True
    ):

        if not patient_id:

            st.error(
                "Please enter a Patient ID."
            )

        elif not get_patient(patient_id):

            st.error(
                f"Patient ID {patient_id} does not exist."
            )

        elif not query.strip():

            st.warning(
                "Please enter your question."
            )

        else:

            st.info(
                f"Query received for {patient_id}"
            )


            question_type = detect_question_type(
                query
            )


            # Determine active agent

            if question_type == "doctor":

                active_agent = "Doctor Agent"

            elif question_type == "laboratory":

                active_agent = "Laboratory Agent"

            elif question_type == "pharmacy":

                active_agent = "Pharmacy Agent"

            elif question_type == "billing":

                active_agent = "Billing Agent"

            elif question_type == "patient":

                active_agent = "Doctor Agent"

            elif question_type == "complete":

                active_agent = "All Agents"

            elif question_type == "cross":

                active_agent = "Multiple Agents"

            else:

                active_agent = "Blackboard"


            st.markdown(
                "### 📡 Blackboard Activity"
            )


            st.write(
                f"🔵 **{active_agent}** is accessing the Blackboard."
            )


            st.write(
                "⬇️ Reading shared knowledge..."
            )


            result = process_query(
                patient_id,
                query
            )


            st.write(
                "⬆️ Blackboard response received."
            )


            # ====================================================
            # DISPLAY SPECIFIC ANSWER
            # ====================================================

            if result["type"] == "error":

                st.error(
                    result["message"]
                )


            elif result["type"] == "patient":

                st.markdown(
                    "### 🧑 Patient Information"
                )

                st.success(
                    result["answer"]
                )


            elif result["type"] == "doctor":

                st.markdown(
                    "### 👨‍⚕️ Doctor Agent Answer"
                )

                st.success(
                    result["answer"]
                )


            elif result["type"] == "laboratory":

                st.markdown(
                    "### 🧪 Laboratory Agent Answer"
                )

                st.success(
                    result["answer"]
                )


            elif result["type"] == "pharmacy":

                st.markdown(
                    "### 💊 Pharmacy Agent Answer"
                )

                st.success(
                    result["answer"]
                )


            elif result["type"] == "billing":

                st.markdown(
                    "### 💰 Billing Agent Answer"
                )

                st.success(
                    result["answer"]
                )


            elif result["type"] == "cross":

                st.markdown(
                    "### 🔄 Multi-Agent Answer"
                )

                st.success(
                    result["answer"]
                )


            elif result["type"] == "complete":

                st.markdown(
                    "### 📋 Complete Blackboard Information"
                )

                st.markdown(
                    result["answer"]
                )

            else:

                st.warning(
                    result["message"]
                )


# ============================================================
# BLACKBOARD
# ============================================================

elif choice == "📋 Blackboard":

    st.header("📋 Central Blackboard")


    st.write(
        "This is the shared knowledge repository. "
        "All agents read from and write to this Blackboard."
    )


    patient_id = st.selectbox(
        "Select Patient",
        [
            p["patient_id"]
            for p in DATA["patients"]
        ]
    )


    board = BLACKBOARD[
        patient_id
    ]


    st.markdown(
        f"""
        <div class="blackboard-box">

        <h2>🧠 BLACKBOARD — {patient_id}</h2>

        <p>
        Last Updated:
        {board["last_updated"]}
        </p>

        </div>
        """,
        unsafe_allow_html=True
    )


    if st.button(
        "🚀 Run Complete Blackboard Process",
        use_container_width=True
    ):

        run_blackboard_process(
            patient_id
        )

        st.success(
            "All agents have read from and written to the Blackboard."
        )

        st.rerun()


    st.markdown("### 🧠 Shared Knowledge")


    tabs = st.tabs([
        "🧑 Patient",
        "👨‍⚕️ Doctor",
        "🧪 Laboratory",
        "💊 Pharmacy",
        "💰 Billing"
    ])


    with tabs[0]:

        st.json(
            board["patient"]
        )


    with tabs[1]:

        if board["doctor"]:

            st.json(
                board["doctor"]
            )

        else:

            st.info(
                "Doctor Agent has not written information yet."
            )


    with tabs[2]:

        if board["laboratory"]:

            st.json(
                board["laboratory"]
            )

        else:

            st.info(
                "Laboratory Agent has not written information yet."
            )


    with tabs[3]:

        if board["pharmacy"]:

            st.json(
                board["pharmacy"]
            )

        else:

            st.info(
                "Pharmacy Agent has not written information yet."
            )


    with tabs[4]:

        if board["billing"]:

            st.json(
                board["billing"]
            )

        else:

            st.info(
                "Billing Agent has not written information yet."
            )


# ============================================================
# DOCTOR AGENT
# ============================================================

elif choice == "👨‍⚕️ Doctor Agent":

    st.header("👨‍⚕️ Doctor Agent")


    st.write(
        "Doctor Agent reads patient information "
        "from the central Blackboard and writes "
        "diagnosis and treatment information back."
    )


    patient_id = st.selectbox(
        "Select Patient",
        [
            p["patient_id"]
            for p in DATA["patients"]
        ]
    )


    if st.button(
        "🩺 Run Doctor Agent",
        use_container_width=True
    ):

        result = doctor_agent(
            patient_id
        )

        st.success(
            "Doctor Agent updated the Blackboard."
        )

        st.json(result)


# ============================================================
# LABORATORY AGENT
# ============================================================

elif choice == "🧪 Laboratory Agent":

    st.header("🧪 Laboratory Agent")


    st.write(
        "Laboratory Agent reads patient information "
        "from the Blackboard and writes laboratory "
        "results to the shared Blackboard."
    )


    patient_id = st.selectbox(
        "Select Patient",
        [
            p["patient_id"]
            for p in DATA["patients"]
        ]
    )


    if st.button(
        "🧪 Run Laboratory Agent",
        use_container_width=True
    ):

        result = laboratory_agent(
            patient_id
        )

        st.success(
            "Laboratory Agent updated the Blackboard."
        )

        st.json(result)


# ============================================================
# PHARMACY AGENT
# ============================================================

elif choice == "💊 Pharmacy Agent":

    st.header("💊 Pharmacy Agent")


    st.write(
        "Pharmacy Agent reads the Doctor Agent's "
        "diagnosis from the Blackboard and writes "
        "medicine information back."
    )


    patient_id = st.selectbox(
        "Select Patient",
        [
            p["patient_id"]
            for p in DATA["patients"]
        ]
    )


    if st.button(
        "💊 Run Pharmacy Agent",
        use_container_width=True
    ):

        result = pharmacy_agent(
            patient_id
        )

        st.success(
            "Pharmacy Agent updated the Blackboard."
        )

        st.json(result)


# ============================================================
# BILLING AGENT
# ============================================================

elif choice == "💰 Billing Agent":

    st.header("💰 Billing Agent")


    st.write(
        "Billing Agent reads Doctor, Laboratory "
        "and Pharmacy information from the "
        "central Blackboard."
    )


    patient_id = st.selectbox(
        "Select Patient",
        [
            p["patient_id"]
            for p in DATA["patients"]
        ]
    )


    if st.button(
        "💰 Run Billing Agent",
        use_container_width=True
    ):

        result = billing_agent(
            patient_id
        )

        st.success(
            "Billing Agent updated the Blackboard."
        )

        st.json(result)


# ============================================================
# AGENT ACTIVITY
# ============================================================

elif choice == "📡 Agent Activity":

    st.header("📡 Agent Activity")


    cursor.execute("""
        SELECT *
        FROM communication
        ORDER BY id DESC
        LIMIT 100
    """)


    rows = cursor.fetchall()


    if rows:

        df = pd.DataFrame(
            rows,
            columns=[
                "ID",
                "Agent",
                "Action",
                "Patient ID",
                "Content",
                "Timestamp",
                "Status"
            ]
        )


        st.dataframe(
            df,
            use_container_width=True,
            hide_index=True
        )

    else:

        st.info(
            "No agent activity recorded."
        )


# ============================================================
# ARCHITECTURE
# ============================================================

elif choice == "🌐 Architecture":

    st.header("🌐 Blackboard Architecture")


    st.markdown("""
    <div class="blackboard-box">

    <h1>🧠 CENTRAL BLACKBOARD</h1>

    <p>
    Shared Knowledge Repository
    </p>

    </div>
    """, unsafe_allow_html=True)


    G = nx.Graph()


    G.add_node(
        "🧠 Blackboard"
    )


    for agent in AGENT_NAMES:

        G.add_node(agent)

        G.add_edge(
            agent,
            "🧠 Blackboard"
        )


    fig, ax = plt.subplots(
        figsize=(11, 8)
    )


    pos = {

        "🧠 Blackboard": (
            0,
            0
        ),

        "Doctor Agent": (
            -2,
            1.5
        ),

        "Laboratory Agent": (
            2,
            1.5
        ),

        "Pharmacy Agent": (
            -2,
            -1.5
        ),

        "Billing Agent": (
            2,
            -1.5
        )

    }


    nx.draw_networkx(
        G,
        pos,
        ax=ax,
        with_labels=True,
        node_size=5000,
        font_size=9,
        font_weight="bold"
    )


    ax.set_title(
        "Blackboard Multi-Agent Architecture"
    )


    ax.axis("off")


    st.pyplot(fig)


    st.markdown("""
    ### How it works

    **1. Agent observes the problem**

    The agent identifies information it needs.

    **2. Agent reads the Blackboard**

    The agent retrieves existing knowledge.

    **3. Agent performs its task**

    The specialized agent processes the information.

    **4. Agent writes the result**

    The updated knowledge is stored on the Blackboard.

    **5. Other agents can use the updated knowledge**

    The Blackboard becomes the shared source of information.
    """)


# ============================================================
# COMMUNICATION HISTORY
# ============================================================

elif choice == "📜 Communication History":

    st.header("📜 Blackboard Communication History")


    cursor.execute("""
        SELECT *
        FROM communication
        ORDER BY id DESC
        LIMIT 100
    """)


    rows = cursor.fetchall()


    if rows:

        df = pd.DataFrame(
            rows,
            columns=[
                "ID",
                "Agent",
                "Action",
                "Patient ID",
                "Content",
                "Timestamp",
                "Status"
            ]
        )


        st.dataframe(
            df,
            use_container_width=True,
            hide_index=True
        )


        st.metric(
            "Total Blackboard Operations",
            len(df)
        )


    else:

        st.info(
            "No communication has been recorded yet."
        )


# ============================================================
# FOOTER
# ============================================================

st.sidebar.markdown("---")


st.sidebar.caption(
    "Blackboard Multi-Agent System"
)


st.sidebar.caption(
    "Python + Streamlit + NetworkX + SQLite"
)

Writing app.py


In [ ]:
%%writefile blackboard_data.json

{
  "patients": [
    {
      "patient_id": "P101",
      "name": "Ravi Kumar",
      "age": 52,
      "gender": "Male",
      "blood_group": "B+",
      "symptoms": "Severe chest pain",
      "city": "Coimbatore"
    },
    {
      "patient_id": "P102",
      "name": "Priya Sharma",
      "age": 35,
      "gender": "Female",
      "blood_group": "O+",
      "symptoms": "High fever and body pain",
      "city": "Chennai"
    },
    {
      "patient_id": "P103",
      "name": "Arun Kumar",
      "age": 28,
      "gender": "Male",
      "blood_group": "A+",
      "symptoms": "Headache and mild fever",
      "city": "Madurai"
    },
    {
      "patient_id": "P104",
      "name": "Meena Raj",
      "age": 46,
      "gender": "Female",
      "blood_group": "AB+",
      "symptoms": "Breathing difficulty",
      "city": "Salem"
    },
    {
      "patient_id": "P105",
      "name": "Karthik S",
      "age": 31,
      "gender": "Male",
      "blood_group": "O-",
      "symptoms": "Stomach pain and vomiting",
      "city": "Trichy"
    }
  ],

  "doctor": [
    {
      "patient_id": "P101",
      "diagnosis": "Possible Cardiac Emergency",
      "treatment": "Immediate cardiac evaluation",
      "severity": "Critical",
      "doctor_name": "Dr. Anand",
      "department": "Cardiology",
      "consultation_status": "Urgent"
    },
    {
      "patient_id": "P102",
      "diagnosis": "Viral Infection",
      "treatment": "Fever control and hydration",
      "severity": "Moderate",
      "doctor_name": "Dr. Meera",
      "department": "General Medicine",
      "consultation_status": "Completed"
    },
    {
      "patient_id": "P103",
      "diagnosis": "Mild Respiratory Infection",
      "treatment": "Rest and symptomatic medication",
      "severity": "Low",
      "doctor_name": "Dr. Kumar",
      "department": "General Medicine",
      "consultation_status": "Follow-up"
    },
    {
      "patient_id": "P104",
      "diagnosis": "Asthma Exacerbation",
      "treatment": "Bronchodilator therapy",
      "severity": "High",
      "doctor_name": "Dr. Priyanka",
      "department": "Pulmonology",
      "consultation_status": "Under Observation"
    },
    {
      "patient_id": "P105",
      "diagnosis": "Gastrointestinal Infection",
      "treatment": "Oral rehydration and medication",
      "severity": "Moderate",
      "doctor_name": "Dr. Suresh",
      "department": "Gastroenterology",
      "consultation_status": "Completed"
    }
  ],

  "laboratory": [
    {
      "patient_id": "P101",
      "test_name": "ECG and Troponin",
      "test_date": "2026-08-31",
      "result": "Abnormal",
      "troponin": "Elevated",
      "blood_pressure": "158/96",
      "test_status": "Urgent"
    },
    {
      "patient_id": "P102",
      "test_name": "Complete Blood Count",
      "test_date": "2026-08-30",
      "result": "Infection Indicators Present",
      "white_blood_cells": "13200 cells/uL",
      "temperature": "39.1 C",
      "test_status": "Completed"
    },
    {
      "patient_id": "P103",
      "test_name": "Blood Profile",
      "test_date": "2026-08-29",
      "result": "Slightly Abnormal",
      "white_blood_cells": "10800 cells/uL",
      "hemoglobin": "13.9 g/dL",
      "test_status": "Reviewed"
    },
    {
      "patient_id": "P104",
      "test_name": "Pulmonary Function Test",
      "test_date": "2026-08-31",
      "result": "Reduced Airflow",
      "oxygen_level": "91%",
      "peak_flow": "280 L/min",
      "test_status": "Monitoring"
    },
    {
      "patient_id": "P105",
      "test_name": "Abdominal Blood Panel",
      "test_date": "2026-08-28",
      "result": "Mild Abnormality",
      "white_blood_cells": "11600 cells/uL",
      "amylase": "118 U/L",
      "test_status": "Completed"
    }
  ],

  "pharmacy": [
    {
      "patient_id": "P101",
      "medicine": "Cardiac Emergency Medication",
      "dosage": "5 mg",
      "quantity": 10,
      "availability": "Available",
      "prescription_status": "Verified",
      "pharmacy_status": "Ready"
    },
    {
      "patient_id": "P102",
      "medicine": "Antipyretic",
      "dosage": "500 mg",
      "quantity": 15,
      "availability": "Available",
      "prescription_status": "Verified",
      "pharmacy_status": "Dispensed"
    },
    {
      "patient_id": "P103",
      "medicine": "Paracetamol",
      "dosage": "650 mg",
      "quantity": 12,
      "availability": "Available",
      "prescription_status": "Approved",
      "pharmacy_status": "Ready"
    },
    {
      "patient_id": "P104",
      "medicine": "Bronchodilator",
      "dosage": "200 mcg",
      "quantity": 8,
      "availability": "Limited Stock",
      "prescription_status": "Verified",
      "pharmacy_status": "Reserved"
    },
    {
      "patient_id": "P105",
      "medicine": "Gastrointestinal Medication",
      "dosage": "20 mg",
      "quantity": 18,
      "availability": "Available",
      "prescription_status": "Approved",
      "pharmacy_status": "Dispensed"
    }
  ],

  "billing": [
    {
      "patient_id": "P101",
      "consultation_fee": 800,
      "laboratory_fee": 2200,
      "medicine_fee": 3500,
      "room_charge": 1500,
      "total_amount": 8000,
      "payment_status": "Pending"
    },
    {
      "patient_id": "P102",
      "consultation_fee": 500,
      "laboratory_fee": 1200,
      "medicine_fee": 650,
      "room_charge": 500,
      "total_amount": 2850,
      "payment_status": "Paid"
    },
    {
      "patient_id": "P103",
      "consultation_fee": 450,
      "laboratory_fee": 900,
      "medicine_fee": 480,
      "room_charge": 300,
      "total_amount": 2130,
      "payment_status": "Paid"
    },
    {
      "patient_id": "P104",
      "consultation_fee": 700,
      "laboratory_fee": 1800,
      "medicine_fee": 1100,
      "room_charge": 900,
      "total_amount": 4500,
      "payment_status": "Pending"
    },
    {
      "patient_id": "P105",
      "consultation_fee": 550,
      "laboratory_fee": 1350,
      "medicine_fee": 750,
      "room_charge": 450,
      "total_amount": 3100,
      "payment_status": "Paid"
    }
  ]
}

Writing blackboard_data.json


In [ ]:
!pkill -f streamlit || true
!pkill -f cloudflared || true

^C
^C


In [ ]:
!ls -lh cloudflared
!./cloudflared --version

ls: cannot access 'cloudflared': No such file or directory
/bin/bash: line 1: ./cloudflared: No such file or directory


In [ ]:
!chmod +x cloudflared

chmod: cannot access 'cloudflared': No such file or directory


In [ ]:
!nohup streamlit run app.py --server.address 0.0.0.0 --server.port 8501 > streamlit.log 2>&1 &

In [ ]:
!sleep 5

In [ ]:
!curl -I http://127.0.0.1:8501

HTTP/1.1 200 OK
date: Thu, 03 Sep 2026 04:49:58 GMT
server: uvicorn
content-type: text/html; charset=utf-8
accept-ranges: bytes
content-length: 7459
last-modified: Thu, 03 Sep 2026 04:49:48 GMT
etag: "f1190b869e5b41fcbbe2ce57dfa8641b"
cache-control: no-cache



In [ ]:
!pwd
!ls -lh cloudflared

/content
ls: cannot access 'cloudflared': No such file or directory


In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!ls -lh cloudflared

-rwxr-xr-x 1 root root 38M Aug 31 10:12 cloudflared


In [ ]:
!pkill -f streamlit || true
!streamlit run app.py --server.address 0.0.0.0 --server.port 8501 > streamlit.log 2>&1 &

^C


In [ ]:
!sleep 5
!cat streamlit.log



2026-09-03 04:50:03.898 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://104.196.70.129:8501



In [ ]:
!./cloudflared tunnel --url http://localhost:8501 > cloudflare.log 2>&1 &

In [ ]:
!sleep 8
!cat cloudflare.log

2026-09-03T04:50:05Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-09-03T04:50:05Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-09-03T04:50:08Z INF +--------------------------------------------------------------------------------------------+
2026-09-03T04:50:08Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-09-03T04:50:08Z INF |  https://surgeon-different-meaning-household.trycloudf